# Día 4 · Unidad 4 — Funciones (I): definición, parámetros y retorno

**Objetivos de aprendizaje**
- Explicar qué es una función y por qué evita repetir código (principio DRY).
- Definir funciones con parámetros posicionales, `keyword` y con valor por defecto.
- Usar `return` correctamente, incluyendo el retorno de varios valores a la vez.
- Documentar funciones con docstrings estilo Google y anotaciones de tipo (`typing`).
- Usar un asistente de IA para refactorizar una función, generarle un docstring y detectar casos borde — sin aceptar el resultado sin comprobarlo.

**Duración estimada:** 100 minutos.

**Requisitos previos:** Día 3 (control de flujo), Día 2 (listas, diccionarios, tuplas).

## 1. Qué es una función

En matemáticas, una función es una relación entre una o más entradas y una salida: $z = f(x, y)$.

En programación, una función es un bloque de código autónomo que encapsula una tarea concreta. Ya has usado funciones sin definirlas tú mismo: `len()`, `sum()`, `any()`...

In [ ]:
primas = [420.0, 310.5, 275.0, 590.25]

print(len(primas))
print(sum(primas))
print(any(prima > 500 for prima in primas))


Para usar estas funciones sólo necesitas conocer su **interfaz**: qué argumentos aceptan y qué devuelven. No necesitas saber cómo está implementado `sum()` por dentro — esa es precisamente la ventaja de trabajar con funciones, las tuyas incluidas.

> 📊 **En Excel esto sería...** como usar una función `LAMBDA` con nombre (Fórmulas → Administrador de nombres → Nueva función LAMBDA): defines la fórmula una única vez y la reutilizas en cualquier celda sin volver a escribirla ni arriesgarte a copiarla mal.

## 2. Por qué definir tus propias funciones: el principio DRY

Supongamos que queremos encontrar el **índice** de la póliza con la prima más alta dentro de una cartera, para poder revisarla.

In [ ]:
primas_cartera_a = [100.0, 250.0, 400.0, 620.0]

idx_max = 0
for idx, prima in enumerate(primas_cartera_a):
    if idx == 0:
        prima_max = prima
    if prima > prima_max:
        idx_max = idx

idx_max


Funciona. Si en otra parte del notebook necesitamos lo mismo para otra cartera, lo más rápido es copiar y pegar el bloque:

In [ ]:
primas_cartera_b = [100.0, 900.0, 400.0, 50.0]

idx_max = 0
for idx, prima in enumerate(primas_cartera_b):
    if idx == 0:
        prima_max = prima
    if prima > prima_max:
        idx_max = idx

idx_max


El resultado es `2`, pero la prima más alta de `primas_cartera_b` está en el índice `1` (900.0). El código no lanza ningún error — simplemente da un resultado incorrecto, que es el tipo de fallo más peligroso porque nadie lo nota si no compara con el resultado esperado.

> ⚠️ **Error típico**: dentro del `if prima > prima_max` sólo se actualiza `idx_max`, pero **no** `prima_max`. Así que `prima_max` se queda congelado en el primer valor de la lista, y a partir de ahí cualquier valor posterior mayor que ese primer valor (aunque ya no sea el máximo real) vuelve a mover `idx_max`. Es exactamente el tipo de fallo silencioso del que avisa `docs/uso_ia_en_el_curso.md`: el código "compila" y "no da error", pero no calcula lo que le pedías.

La versión corregida actualiza también `prima_max` dentro del `if`:

In [ ]:
idx_max = 0
for idx, prima in enumerate(primas_cartera_b):
    if idx == 0:
        prima_max = prima
    if prima > prima_max:
        idx_max = idx
        prima_max = prima

idx_max


Ahora sí: `1`. Pero si copiamos y pegamos este bloque en diez sitios del notebook, y luego encontramos el fallo, hay que corregirlo en los diez sitios. Esto es justo lo que evita el principio **DRY** (*Don't Repeat Yourself*): escribir la lógica una vez, dentro de una función, y reutilizarla.

## 3. Declaración y llamada de una función

```python
def <nombre_funcion>(<parámetros>):
    <sentencias>
    return <valor>
```

- `def`: palabra reservada que declara una función.
- `nombre_funcion`: un identificador válido de Python.
- `parámetros`: lista opcional de parámetros separados por comas.
- `sentencias`: el cuerpo de la función.

In [ ]:
def indice_prima_maxima(primas):
    idx_max = 0
    for idx, prima in enumerate(primas):
        if idx == 0:
            prima_max = prima
        if prima > prima_max:
            idx_max = idx
            prima_max = prima
    return idx_max


print(indice_prima_maxima(primas_cartera_a))
print(indice_prima_maxima(primas_cartera_b))


Ahora la lógica vive en un único lugar. Si mañana encontramos otro fallo, lo arreglamos una vez y todas las llamadas se benefician.

## 4. Parámetros

### Parámetros posicionales

La forma más simple de pasar datos a una función: en la llamada, los argumentos se vinculan a los parámetros **en el orden** en que aparecen.

In [ ]:
def calcular_prima_total(numero_polizas, prima_unitaria, producto):
    print(f"Prima total para {numero_polizas} pólizas de {producto}: {numero_polizas * prima_unitaria}")


calcular_prima_total(50, 320.0, "auto")


Nada impide pasar los argumentos posicionales en el orden equivocado — Python no sabe que el segundo argumento "debería" ser una prima:

In [ ]:
calcular_prima_total(50, "auto", 320.0)


No hay ningún error: `50 * "auto"` es una operación válida en Python (repite el string 50 veces), así que el resultado es una cadena kilométrica en vez de un número. Con argumentos posicionales, el orden en la llamada debe coincidir con el orden en la definición.

También debe coincidir el **número** de argumentos:

In [ ]:
try:
    calcular_prima_total(50, 320.0)
except TypeError as e:
    print(type(e).__name__, "-", e)


### Argumentos `keyword`

Si en la llamada especificas `nombre_parametro=valor`, el orden deja de importar:

In [ ]:
calcular_prima_total(producto="auto", numero_polizas=50, prima_unitaria=320.0)


### Parámetros con valor por defecto

Si en la definición un parámetro tiene la forma `nombre=valor`, ese `valor` se usa cuando la llamada no lo especifica.

In [ ]:
def calcular_prima_total(numero_polizas, prima_unitaria, producto="vida"):
    print(f"Prima total para {numero_polizas} pólizas de {producto}: {numero_polizas * prima_unitaria:.2f}")


calcular_prima_total(10, 480.0)
calcular_prima_total(10, 480.0, producto="hogar")


Los parámetros con valor por defecto siempre deben ir **después** de los que no lo tienen. Este código no compila:

```python
def calcular_prima_total(numero_polizas=1, prima_unitaria, producto):
    ...
```

Si lo pruebas, Python lanza `SyntaxError: parameter without a default follows parameter with a default` — ni siquiera llega a ejecutarse, falla al leer el código.

**Para ti:** escribe una función `calcular_recargo(prima, recargo_pct=0.05)` que devuelva la prima incrementada en ese porcentaje. Pruébala una vez sin indicar `recargo_pct` y otra vez indicando, por ejemplo, `recargo_pct=0.12`.

> ⚠️ **Error típico**: un valor por defecto **mutable** (una lista, un diccionario...) se crea **una sola vez**, cuando Python lee la definición de la función — no en cada llamada. Si lo modificas dentro de la función, el cambio persiste entre llamadas distintas, que es casi nunca lo que quieres.

In [ ]:
def anadir_siniestro(siniestro, historial=[]):
    historial.append(siniestro)
    return historial


print(anadir_siniestro("SIN-2024-001"))
print(anadir_siniestro("SIN-2024-002"))


La segunda llamada "hereda" el siniestro de la primera, aunque no se lo hayamos pasado — porque ambas llamadas comparten la **misma lista** `historial`, creada una única vez. La forma correcta es usar `None` como valor por defecto y crear la lista nueva dentro de la función:

In [ ]:
def anadir_siniestro(siniestro, historial=None):
    if historial is None:
        historial = []
    historial.append(siniestro)
    return historial


print(anadir_siniestro("SIN-2024-001"))
print(anadir_siniestro("SIN-2024-002"))


## 5. La cláusula `return`

`return` termina la ejecución de la función de inmediato y "devuelve" un valor a quien la llamó. No es lo mismo que `print()`: `print` sólo muestra algo en pantalla, `return` permite **guardar** el resultado en una variable para seguir usándolo.

In [ ]:
def ultima_prima(historial_primas):
    return historial_primas[-1]


x = ultima_prima([310.0, 295.0, 340.0])
print(f"x = {x}")


In [ ]:
def ultima_prima_con_print(historial_primas):
    print(historial_primas[-1])


x = ultima_prima_con_print([310.0, 295.0, 340.0])
print(f"x = {x}")


`ultima_prima_con_print` **muestra** el valor por pantalla, pero como no hace `return`, la función devuelve `None` por defecto — y `x` termina valiendo `None`, no `340.0`.

> ⚠️ **Error típico**: olvidar el `return`. Es uno de los fallos más comunes al empezar con funciones: el cuerpo de la función calcula el resultado correcto, lo imprime incluso, pero se te olvida devolverlo — y quien llama a la función recibe `None` sin ningún error visible.

In [ ]:
def calcular_prima_pura_con_bug(frecuencia, severidad):
    prima_pura = frecuencia * severidad
    print(f"Prima pura calculada: {prima_pura:.2f}")
    # falta el return


resultado = calcular_prima_pura_con_bug(0.08, 450.0)
print("Lo que ha recibido quien llamó a la función:", resultado)


In [ ]:
def calcular_prima_pura(frecuencia, severidad):
    prima_pura = frecuencia * severidad
    return prima_pura


resultado = calcular_prima_pura(0.08, 450.0)
print("Ahora sí:", resultado)


Todas las sentencias posteriores a un `return` que se ejecuta no se llegan a ejecutar — la función termina ahí:

In [ ]:
def dividir_siniestralidad(siniestros, primas):
    if primas == 0:
        print("No se puede dividir entre cero primas")
        return None
    print("Calculando ratio de siniestralidad...")
    return siniestros / primas


print(dividir_siniestralidad(180.0, 300.0))
print(dividir_siniestralidad(180.0, 0))


### `return` y *tuple unpacking*

Ya sabes iterar sobre una lista de tuplas y desempaquetar sus valores:

In [ ]:
frec_sev = [(0.01, 200), (0.15, 300), (0.05, 400)]

for frec, sev in frec_sev:
    print(f"Frecuencia {frec} y severidad {sev} -> prima de riesgo {frec * sev:.2f}")


De la misma forma, una función puede devolver una tupla, y quien la llama puede desempaquetarla directamente en varias variables:

In [ ]:
def combinacion_prima_maxima(frec_sev):
    """Encuentra la combinación (frecuencia, severidad) con mayor prima de riesgo.

    Args:
        frec_sev: lista de tuplas (frecuencia, severidad).

    Returns:
        tuple[float, float]: la frecuencia y la severidad que producen
        la mayor prima de riesgo (frecuencia * severidad).
    """
    prima_max = 0.0
    frec_max, sev_max = frec_sev[0]
    for frec, sev in frec_sev:
        prima_riesgo = frec * sev
        if prima_riesgo > prima_max:
            prima_max = prima_riesgo
            frec_max, sev_max = frec, sev
    return frec_max, sev_max


frec, sev = combinacion_prima_maxima(frec_sev)
print(f"Frecuencia: {frec}, severidad: {sev}")


**Para ti:** añade a la lista `frec_sev` una combinación con la misma prima de riesgo que la actual ganadora (por ejemplo, `(0.03, 500)` si la ganadora tiene prima de riesgo 45). Ejecuta `combinacion_prima_maxima` de nuevo: ¿qué combinación devuelve en caso de empate? Comprueba si el comportamiento tiene sentido para ti (¿debería quedarse con la primera que encuentra, o con la última?).

## 6. Docstrings

Un docstring documenta qué hace una función, justo debajo de la línea `def`. Compara: sin documentación, es difícil saber qué calcula esta función sólo con leer el código.

In [ ]:
def contiene_patron(historial, patron):
    idx = 0
    for banda in historial:
        if idx == len(patron):
            break
        if patron[idx] == banda:
            idx += 1
    return idx == len(patron)


¿Qué hace `contiene_patron`? Sin más contexto, no es evidente. Añadamos un docstring estilo [Google](https://sphinxcontrib-napoleon.readthedocs.io/en/latest/example_google.html), que es el que usaremos durante el resto del curso:

In [ ]:
def contiene_patron(historial, patron):
    """Comprueba si `patron` aparece, en orden, dentro de `historial`.

    Se usa por ejemplo para detectar si un asegurado ha pasado por una
    secuencia concreta de bandas de severidad de siniestro (no
    necesariamente en posiciones consecutivas, pero sí en ese orden).

    Args:
        historial: lista de bandas de severidad en orden cronológico,
            p. ej. ["BAJA", "MEDIA", "ALTA", "BAJA", "ALTA"].
        patron: secuencia de bandas que se busca dentro de `historial`,
            p. ej. ["MEDIA", "ALTA"].

    Returns:
        bool: True si `patron` aparece en orden dentro de `historial`.
    """
    idx = 0
    for banda in historial:
        if idx == len(patron):
            break
        if patron[idx] == banda:
            idx += 1
    return idx == len(patron)


historial_asegurado = ["BAJA", "MEDIA", "ALTA", "BAJA", "ALTA"]
print(contiene_patron(historial_asegurado, ["MEDIA", "ALTA"]))
print(contiene_patron(historial_asegurado, ["ALTA", "MEDIA"]))


A partir de esta unidad, toda función que escribas en el curso debe llevar docstring — es la convención fijada en `CLAUDE.md`.

## 7. Anotaciones de tipo (`typing`)

Python no obliga a declarar el tipo de los parámetros ni del valor de retorno, pero puedes **anotarlos**. No cambian el comportamiento en tiempo de ejecución (Python no los comprueba por sí solo), pero documentan la intención y permiten que tu editor y herramientas como `mypy` detecten errores antes de ejecutar nada.

In [ ]:
def contiene_patron(historial: list[str], patron: list[str]) -> bool:
    """Comprueba si `patron` aparece, en orden, dentro de `historial`.

    Args:
        historial: lista de bandas de severidad en orden cronológico.
        patron: secuencia de bandas que se busca dentro de `historial`.

    Returns:
        bool: True si `patron` aparece en orden dentro de `historial`.
    """
    idx = 0
    for banda in historial:
        if idx == len(patron):
            break
        if patron[idx] == banda:
            idx += 1
    return idx == len(patron)


contiene_patron(historial_asegurado, ["MEDIA", "ALTA"])


> 📊 **En Excel esto sería...** como restringir una celda con Validación de datos a "sólo números": no impide que alguien fuerce un valor incorrecto por otra vía, pero deja claro para cualquiera que abra el archivo qué tipo de dato se espera ahí.

Nota: en Python moderno (3.9+, y desde luego en la versión 3.14 de este entorno) se usan los tipos genéricos incorporados como `list[str]` o `dict[str, float]` directamente. En código más antiguo verás `from typing import List, Dict` y `List[str]` — es equivalente, pero ya no hace falta.

**Para ti:** vuelve a la función `indice_prima_maxima` que escribiste en la sección 2 y añádele un docstring estilo Google y anotaciones de tipo (`list[float] -> int`).

## 8. Con ayuda de la IA: refactorizar, documentar y encontrar casos borde

Imagina que heredas esta función en un notebook de un compañero, sin más contexto que el nombre del archivo (`pricing.py`). Es código que funciona — el negocio lleva tiempo usando estos números — pero es difícil de entender y de mantener:

In [ ]:
def calc(a, b, c):
    if c == 1:
        r = a * b
    elif c == 2:
        r = a * b * 0.9
    elif c == 3:
        r = a * b * 1.2
    else:
        r = a * b
    return r


`a` es el número de pólizas, `b` es la prima unitaria, `c` es un código de descuento/recargo (1 = sin ajuste, 2 = descuento del 10%, 3 = recargo del 20%). Nombres de una letra, números "mágicos" sin explicar, sin docstring: buena candidata a refactorización asistida.

**Antes de pedirle nada a la IA**, fijamos un caso de prueba con el comportamiento **actual** — el refactor no puede cambiar estos resultados, aunque el nombre de la función sí:

In [ ]:
casos_prueba = [(50, 320.0, 1), (50, 320.0, 2), (50, 320.0, 3), (50, 320.0, 9)]
resultados_antes = [calc(a, b, c) for a, b, c in casos_prueba]
print(resultados_antes)


### Prompt sugerido

In [ ]:
prompt_refactor = """
Contexto: tengo esta funcion de calculo de primas en un notebook de Python,
heredada sin documentacion:

def calc(a, b, c):
    if c == 1:
        r = a * b
    elif c == 2:
        r = a * b * 0.9
    elif c == 3:
        r = a * b * 1.2
    else:
        r = a * b
    return r

a es el numero de polizas, b es la prima unitaria, y c es un codigo de
descuento (1 = sin ajuste, 2 = descuento del 10%, 3 = recargo del 20%).

Objetivo: quiero que refactorices la funcion con nombres claros, que le
anadas un docstring estilo Google explicando que hace, sus parametros y
que devuelve, con type hints, y que me sugieras que casos borde no estoy
contemplando (por ejemplo codigos de descuento no previstos, o valores
negativos de a o b).

Restricciones: el comportamiento para los codigos 1, 2 y 3 debe ser
exactamente el mismo que el original (mismos numeros de salida). No
cambies la logica de negocio, solo la forma.

Formato de salida: dame la funcion refactorizada completa, mas una lista
corta de los casos borde que sugieres revisar.
"""
print(prompt_refactor)


**Qué se espera obtener**
- La misma lógica, con nombres descriptivos (`numero_polizas`, `prima_unitaria`, `codigo_descuento`).
- Un docstring estilo Google (`Args` / `Returns`).
- Anotaciones de tipo.
- Una lista de casos borde no contemplados hoy — por ejemplo: `codigo_descuento` desconocido (¿debería fallar o simplemente no aplicar ajuste, como hace ahora el `else`?), `numero_polizas` negativo o cero, `prima_unitaria` negativa.

**Qué debes verificar antes de darlo por bueno**
- Que el refactor produce **exactamente los mismos resultados** que la función original para los mismos casos de prueba — no sólo que "se vea mejor". Esto se comprueba ejecutando ambas versiones con los mismos argumentos y comparando, nunca leyendo el código y dando por hecho que es equivalente.
- Que los casos borde que sugiere tienen sentido de negocio real — eso lo decides tú con tu criterio actuarial, la IA no sabe si un `codigo_descuento` desconocido debería rechazarse o ignorarse.
- Que el docstring describe lo que el código realmente hace, no lo que "debería" hacer.

Una posible respuesta (escrita aquí a mano para que puedas ejecutar el notebook sin depender de un asistente externo; en clase, genera la tuya con Claude Code y compárala):

In [ ]:
def calcular_prima_total(numero_polizas: int, prima_unitaria: float, codigo_descuento: int) -> float:
    """Calcula la prima total de una cartera aplicando un ajuste por código.

    Args:
        numero_polizas: número de pólizas de la cartera.
        prima_unitaria: prima de una única póliza.
        codigo_descuento: 1 = sin ajuste, 2 = descuento del 10%,
            3 = recargo del 20%. Cualquier otro código no aplica ajuste
            (se conserva el comportamiento de la función original).

    Returns:
        float: prima total (numero_polizas * prima_unitaria * factor).
    """
    factores_por_codigo = {1: 1.0, 2: 0.9, 3: 1.2}
    factor = factores_por_codigo.get(codigo_descuento, 1.0)
    return numero_polizas * prima_unitaria * factor


resultados_despues = [calcular_prima_total(a, b, c) for a, b, c in casos_prueba]
print(resultados_despues)

assert resultados_despues == resultados_antes, "El refactor ha cambiado el resultado de algún caso conocido"
print("Comportamiento verificado: los resultados coinciden con la función original.")


El `assert` es la parte que de verdad importa del ejercicio: confirma que el refactor **no** cambió el resultado para ningún caso conocido, tal como pedía la restricción del prompt. Fíjate también en un matiz que sólo puedes decidir tú: el `codigo_descuento` desconocido (`9` en el ejemplo) sigue sin aplicar ningún ajuste, igual que en el `else` original — la IA puede señalarlo como caso borde, pero decidir si eso es correcto o si debería lanzar un error es un criterio de negocio, no técnico.

## 9. Mini-proyecto: el juego del trile

Combinando parámetros, `return`, docstrings y tipado, construimos una versión sencilla y sin trampa del [trile](https://es.wikipedia.org/wiki/Trile), en cuatro funciones pequeñas.

In [ ]:
import random

def revolver_tablero(tablero: list[str]) -> list[str]:
    """Ordena de manera aleatoria el tablero.

    Args:
        tablero: representación del tablero de juego.

    Returns:
        list[str]: el mismo tablero, con las posiciones mezcladas.
    """
    random.shuffle(tablero)
    return tablero


In [ ]:
def adivinar(respuesta: str | None = None) -> int:
    """Pide (o recibe) la posición elegida por el jugador.

    En una sesión interactiva, si no se pasa `respuesta`, la pide por
    teclado con `input()`. Para poder ejecutar este notebook de principio
    a fin sin bloquearse esperando teclado, también admite una
    `respuesta` ya decidida (así se prueba el resto de la lógica).

    Args:
        respuesta: posición elegida como texto ("0", "1" o "2"). Si es
            None, se pide por teclado.

    Returns:
        int: la posición elegida, como entero.
    """
    posicion = respuesta
    while posicion not in ("0", "1", "2"):
        posicion = input("Escoge un número: 0, 1 o 2: ")
    return int(posicion)


In [ ]:
def verificar(tablero: list[str], posicion: int) -> None:
    """Comprueba si la posición elegida contiene la bolita.

    Args:
        tablero: representación del tablero de juego, ya mezclado.
        posicion: posición elegida por el jugador.

    Returns:
        None
    """
    if tablero[posicion] == "O":
        print("¡Correcto, has ganado!", tablero)
    else:
        print("¡Fallaste, has perdido!", tablero)


In [ ]:
def jugar(respuesta: str | None = None) -> None:
    """Ejecuta una partida completa del trile.

    Args:
        respuesta: ver `adivinar`. Se pasa fija aquí para poder demostrar
            el juego sin depender de teclado; en una sesión interactiva
            real se llamaría `jugar()` sin argumentos.
    """
    tablero = ["", "O", ""]
    tablero = revolver_tablero(tablero)
    posicion = adivinar(respuesta)
    verificar(tablero, posicion)


random.seed(42)  # fija el resultado del shuffle para que el notebook sea reproducible
jugar(respuesta="1")


**Para ti:** modifica `jugar` para que reciba también el tamaño del tablero (número de casillas, hoy fijo a 3) como parámetro con valor por defecto `3`, y ajusta `revolver_tablero`/`adivinar` en consecuencia.

## Resumen

- Una función encapsula una tarea para no repetir código (principio DRY); su interfaz son sus parámetros y su valor de retorno.
- Los parámetros pueden ser posicionales, `keyword` o con valor por defecto — los que tienen valor por defecto siempre van al final de la definición.
- Nunca uses una lista, un diccionario o cualquier otro objeto mutable como valor por defecto: se crea una única vez y se comparte entre llamadas.
- `return` devuelve un valor a quien llama a la función; sin `return`, la función devuelve `None` aunque haya impreso algo por pantalla.
- Una función puede devolver varios valores a la vez como una tupla, y desempaquetarlos en la llamada.
- Los docstrings (estilo Google) y las anotaciones de tipo documentan qué hace una función, qué espera y qué devuelve — a partir de esta unidad, son obligatorios en el código del curso.
- Un asistente de IA puede refactorizar, documentar y sugerir casos borde muy rápido — pero el refactor sólo se acepta si un caso de prueba confirma que el comportamiento no ha cambiado, y los casos borde sólo se aceptan si tienen sentido de negocio para ti.

**Siguiente:** `02_teoria_args_kwargs.ipynb`, para pasar un número variable de argumentos a una función.